# 📰 News RSS → Podcast MVP — Qwen3-TTS

Automatically fetch real news from RSS feeds and generate a spoken morning briefing.

### Supported RSS Feeds
| Source | Feed URL |
| --- | --- |
| TechCrunch | https://techcrunch.com/feed/ |
| Hacker News | https://news.ycombinator.com/rss |
| BBC Technology | http://feeds.bbci.co.uk/news/technology/rss.xml |
| MIT Tech Review | https://www.technologyreview.com/feed/ |


In [ ]:
!pip install -q qwen-tts soundfile feedparser requests beautifulsoup4
import os
import torch
import feedparser
import soundfile as sf
from bs4 import BeautifulSoup
from IPython.display import Audio, display
from qwen_tts import Qwen3TTSModel


In [ ]:
MODEL_SIZE = "1.7B"
OUTPUT_DIR = "podcast_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RSS_FEEDS = {
    "Hacker News": "https://news.ycombinator.com/rss",
    "TechCrunch": "https://techcrunch.com/feed/",
    "BBC Technology": "http://feeds.bbci.co.uk/news/technology/rss.xml",
    "MIT Tech Review": "https://www.technologyreview.com/feed/"
}


In [ ]:
SELECTED_FEED = "BBC Technology"  # Change this to any feed from RSS_FEEDS
feed_url = RSS_FEEDS[SELECTED_FEED]

print(f"Fetching news from {SELECTED_FEED}...")
feed = feedparser.parse(feed_url)

top_stories = []
for entry in feed.entries[:5]:
    title = entry.title
    summary = ""
    if hasattr(entry, 'summary'):
        soup = BeautifulSoup(entry.summary, "html.parser")
        summary = soup.get_text().strip()
    
    top_stories.append({'title': title, 'summary': summary})
    print(f"- {title}")


In [ ]:
def write_briefing_script(stories, source_name):
    script = f"Good morning! Here's your tech briefing from {source_name}.\n\n"
    
    for i, story in enumerate(stories):
        if i == 0:
            script += f"Our top story today: {story['title']}. "
        else:
            script += f"In other news: {story['title']}. "
            
        if story['summary']:
            script += f"{story['summary']} "
        
        script += "\n\n"
        
    script += "That's all for now. Stay tuned for more updates, and have a great day ahead!"
    return script

script_text = write_briefing_script(top_stories, SELECTED_FEED)
print("\n--- SCRIPT ---\n")
print(script_text)


In [ ]:
print("Loading VoiceDesign Model...")
model = Qwen3TTSModel.from_pretrained(
    f"Qwen/Qwen3-TTS-12Hz-{MODEL_SIZE}-VoiceDesign",
    device_map="cuda:0",
    dtype=torch.bfloat16,
    attn_implementation="sdpa"
)

INSTRUCT = "A confident, authoritative news anchor voice — clear diction, measured pace, professional gravitas, like a BBC World Service announcer"

print("Generating audio briefing (this might take a minute)...")
audio = model.generate_voice_design(
    text=script_text.replace('\n', ' '),
    language="english",
    instruct=INSTRUCT
)

out_path = f"{OUTPUT_DIR}/morning_briefing.wav"
sf.write(out_path, audio, samplerate=24000)
print("Saved briefing!")


In [ ]:
print("Morning Briefing from", SELECTED_FEED)
display(Audio(out_path))
print("\nTranscript:")
print(script_text)


## ⏱️ Scheduling this Podcast Generator

To run this notebook on a recurring schedule and get a daily podcast:
1. **Colab Pro Scheduled Runs:** If you have Colab Pro, you can use the "Schedule" button in the top right to run this daily.
2. **GitHub Actions:** You can export this as a Python script and run it on GitHub Actions with a CRON trigger (using a Replicate or local GPU runner).


In [ ]:
from google.colab import files
files.download(out_path)
